# Transformar Data de Carreras

1. Leer la tabla bronze `races`
1. Mantener solo las columnas requeridas para analítica (Dropear columna `url`)
1. Estandarizar los nombres de la columnas a  snake_case y traducir a español (`raceName` → `nombre_carrera`, `circuitId` → `id_circuito`)
1. Renombrar columnas hacia un nombre más significativa (`date` → `fecha carrera`)
1. Remover records duplicados
1. Transformar los valores de la columna `nombre carrera` a Title Case
1. Escribir los valores tranformados a tabla silver `carreras` 



#### 

![Formula1 Raw Data.png](../../z-course-images/formula1-raw-data-erd.png "Formula1 Raw Data.png")

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/03.Configuracion_Ambiente

In [0]:
tabla_bronze = f"{catalogo}.{esquema_bronze}.races"
tabla_silver = f"{catalogo}.{esquema_silver}.carreras"

In [0]:
from pyspark.sql import functions as F

#### 1 - Leer la tabla bronze `races`

In [0]:
df_races = spark.table(tabla_bronze )

#### 2 - Mantener solo las columnas requeridas para analítica (Dropear columna `url`)

In [0]:
df_carreras_sel = df_races.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("ingesta_timestamp"),
    F.col("archivo_fuente")
)

#### 3 & 4 - Estandarizar los nombres de la columnas
- Estandarizar los nombres de la columnas a  snake_case y traducir a español (`raceName` → `nombre_carrera`, `circuitId` → `id_circuito`)
- Renombrar columnas hacia un nombre más significativa (`date` → `fecha carrera`)

In [0]:
df_carreras_renommbre = (
    df_carreras_sel
        .withColumnsRenamed({
            "circuitId": "id_circuito",
            "season": "temporada",
            "round": "numero_carrera",
            "raceName": "nombre_carrera",
            "date": "fecha_carrera"
        })
)

In [0]:
display(df_carreras_renommbre)

#### 5 - Remover records duplicados

In [0]:
df_carreras_distinct = df_carreras_renommbre.dropDuplicates(["temporada","numero_carrera"])

In [0]:
display(df_carreras_distinct)

#### 6 - Transformar los valores de la columna `nombre carrera` a Title Case

In [0]:
df_carreras_final = (
    df_carreras_distinct
        .withColumn('nombre_carrera', F.initcap(F.col("nombre_carrera")))
)

In [0]:
display(df_carreras_final)

#### 7 - Escribir los valores tranformados a tabla silver `carreras`

In [0]:
(
    df_carreras_final
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabla_silver)
)

In [0]:
display(spark.table(tabla_silver))